# Certificación Profesional en Desarrollador de Agentes de IA
## Clase 3 — Salidas estructuradas y aplicaciones robustas

ITBA Educación Ejecutiva

Generar resultados que otros sistemas puedan consumir de forma confiable — no texto libre que hay que parsear a mano y cruzar los dedos.

### Cómo se usa este notebook

- **Durante la clase en vivo**: el docente corre y explica cada celda (demo con AWS Bedrock). Los alumnos no programan en simultáneo.
- **Celdas 🏠 Practicá**: tarea individual para después de la clase.
- **Sección 9 (Laboratorio)**: tarea principal de la semana — checkpoint del proyecto integrador. Se resuelve con el proveedor de cada uno (Bedrock, Anthropic, OpenAI, Google u Ollama), no hace falta que sea Bedrock.

## Objetivo de la clase

Al final de la clase vas a poder:
- Definir un JSON Schema y un modelo Pydantic equivalente para validar salidas de un LLM.
- Obtener salidas estructuradas usando el mecanismo nativo del proveedor cuando existe, y un patrón de reintento universal cuando no.
- Usar tool calling para que el modelo pida ejecutar una función externa con argumentos validados.
- Configurar timeouts, reintentos con backoff+jitter, y registrar tokens/costo/latencia de cada llamada.
- Consumir una respuesta en streaming.
- Refactorizar el cliente de la Clase 2 detrás de una interfaz común (`LLMProvider`), agnóstica de proveedor.

**Checkpoint del proyecto integrador de hoy**: servicio que devuelve salidas estructuradas y validadas (Pydantic) a partir de texto libre, sobre el cliente ya agnóstico de proveedor (ver `proyecto-integrador/README.md`, checkpoint Clase 3).

## Agenda

1. JSON Schema y Pydantic
2. Structured outputs (mecanismo nativo del proveedor)
3. Manejo de respuestas inválidas (patrón universal de reintento)
4. Tool calling
5. Timeouts, reintentos y backoff
6. Observabilidad básica: tokens, costo y latencia
7. Streaming
8. Abstracción de proveedores
9. Laboratorio: servicio de extracción estructurada

## 0. Punto de partida

Recordá las funciones `llamar_bedrock`, `llamar_anthropic`, `llamar_openai`, `llamar_google` y `llamar_ollama` de la Clase 2 — hoy las vamos a envolver en clases con una interfaz común. Para que este notebook sea autocontenido, las volvemos a definir acá (son las mismas que ya tenías).

In [ ]:
import os
import json
import time
import random
from dotenv import load_dotenv

load_dotenv()
provider = os.environ.get("LLM_PROVIDER", "google")
print(f"Proveedor configurado: {provider}")

Proveedor configurado: google


In [2]:
from google import genai
from google.genai import types

cliente = genai.Client(
    api_key=os.environ["GOOGLE_API_KEY"],
    http_options=types.HttpOptions(timeout=30000),
)
model_id = os.environ.get("GOOGLE_MODEL_ID", "gemini-3.5-flash-lite")

def llamar_google(prompt, system=None, temperature=0.7, max_tokens=500):
    config = types.GenerateContentConfig(
        system_instruction=system,
        temperature=temperature,
        max_output_tokens=max_tokens,
    )
    respuesta = cliente.models.generate_content(
        model=model_id,
        contents=prompt,
        config=config,
    )
    return respuesta.text

In [3]:
print(llamar_google("Decí hola"))
print(llamar_google("¿Quién sos?", system="Sos un pirata. Respondé en una sola oración."))

¡Hola! ¿Cómo estás?
¡Soy el temido Capitán Barbanegra, y navego los Siete Mares en busca de ron y tesoros olvidados!


## 1. JSON Schema y Pydantic

- **JSON Schema**: un documento (también JSON) que describe la forma esperada de otro documento JSON — tipos, campos requeridos, enums, etc. Es el estándar que entienden las APIs de los proveedores para "forzar" un formato de salida.
- **Pydantic**: librería de Python que define esa misma forma como una clase, valida datos contra ella, y puede generar el JSON Schema automáticamente — no hace falta escribirlo dos veces.

In [34]:
# %pip install pydantic  # si hace falta instalarlo

from pydantic import BaseModel, Field, ValidationError
from typing import Literal

class ConsultaSoporte(BaseModel):
    categoria: Literal["facturacion", "tecnico", "cuenta", "otro"]
    urgencia: Literal["baja", "media", "alta"]
    resumen: str = Field(..., max_length=200, description="Resumen breve del reclamo, en una oración")
    requiere_escalamiento: bool = Field(default=False, description="True si el reclamo requiere atención inmediata, si lleva 24hs, si pide hablar con un supervisor")

# El JSON Schema se genera solo a partir de la clase:
schema = ConsultaSoporte.model_json_schema()
print(json.dumps(schema, indent=2, ensure_ascii=False))

{
  "properties": {
    "categoria": {
      "enum": [
        "facturacion",
        "tecnico",
        "cuenta",
        "otro"
      ],
      "title": "Categoria",
      "type": "string"
    },
    "urgencia": {
      "enum": [
        "baja",
        "media",
        "alta"
      ],
      "title": "Urgencia",
      "type": "string"
    },
    "resumen": {
      "description": "Resumen breve del reclamo, en una oración",
      "maxLength": 200,
      "title": "Resumen",
      "type": "string"
    },
    "requiere_escalamiento": {
      "default": false,
      "description": "True si el reclamo requiere atención inmediata, si lleva 24hs, si pide hablar con un supervisor",
      "title": "Requiere Escalamiento",
      "type": "boolean"
    }
  },
  "required": [
    "categoria",
    "urgencia",
    "resumen"
  ],
  "title": "ConsultaSoporte",
  "type": "object"
}


In [5]:
# Validación: un dict válido pasa, uno inválido lanza ValidationError con el detalle exacto de qué falló
dato_valido = {"categoria": "tecnico", "urgencia": "alta", "resumen": "El servicio no responde desde hace 2 horas."}
consulta = ConsultaSoporte(**dato_valido)
print(consulta)

dato_invalido = {"categoria": "inventado", "urgencia": "alta", "resumen": "El servicio no responde desde hace 2 horas."}

try:
    ConsultaSoporte(**dato_invalido)
except ValidationError as e:
    print("\nError de validación (esperado):")
    print(e)

categoria='tecnico' urgencia='alta' resumen='El servicio no responde desde hace 2 horas.' requiere_escalamiento=False

Error de validación (esperado):
1 validation error for ConsultaSoporte
categoria
  Input should be 'facturacion', 'tecnico', 'cuenta' or 'otro' [type=literal_error, input_value='inventado', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/literal_error


**🏠 Practicá (tarea, después de la clase)**: agregá un campo `requiere_escalamiento: bool` al modelo `ConsultaSoporte`. Regenerá el JSON Schema y confirmá que el nuevo campo aparece en `properties`. Probá instanciar el modelo sin ese campo — ¿qué pasa? (pista: Pydantic trata los campos sin default como requeridos).

## 2. Structured outputs

Pedirle al modelo "devolveme JSON" en el prompt **no garantiza** que la salida sea JSON válido, ni que cumpla el schema — el modelo puede agregar texto antes/después, usar comillas simples, olvidar un campo, etc.

AWS Bedrock (igual que OpenAI y algunos otros proveedores) tiene un mecanismo **nativo** que fuerza al modelo a producir una salida que cumple exactamente el schema: en Bedrock, el campo `outputConfig.textFormat` de la Converse API.

In [ ]:
# BEDROCK — solo referencia, NO CORRER (versión Google más abajo)

schema_bedrock = ConsultaSoporte.model_json_schema()
schema_bedrock["additionalProperties"] = False  # requerido por el validador de Bedrock para objetos "cerrados"

texto_libre = "Hace dos días que no puedo entrar a mi cuenta y ya perdí una reunión importante por esto, necesito que lo resuelvan YA."

response = bedrock.converse(
    modelId=model_id,
    messages=[{"role": "user", "content": [{"text": f"Clasificá esta consulta de soporte: '{texto_libre}'"}]}],
    outputConfig={
        "textFormat": {
            "type": "json_schema",
            "structure": {
                "jsonSchema": {
                    "name": "consulta_soporte",
                    "description": "Clasificación estructurada de una consulta de soporte",
                    "schema": json.dumps(schema_bedrock),
                }
            }
        }
    },
)

raw_json = response["output"]["message"]["content"][0]["text"]
print("JSON crudo devuelto por el modelo:", raw_json)

consulta_validada = ConsultaSoporte.model_validate_json(raw_json)
print("\nObjeto Python validado:", consulta_validada)

In [6]:
texto_libre = "Hola, quería consultar si la factura de este mes ya está disponible, gracias."

config = types.GenerateContentConfig(
    response_mime_type="application/json",
    response_schema=ConsultaSoporte,
)

respuesta = cliente.models.generate_content(
    model=model_id,
    contents=f"Clasificá esta consulta de soporte: '{texto_libre}'",
    config=config,
)

raw_json = respuesta.text
print("JSON crudo devuelto por el modelo:", raw_json)

consulta_validada = ConsultaSoporte.model_validate_json(raw_json)
print("\nObjeto Python validado:", consulta_validada)

JSON crudo devuelto por el modelo: {
  "categoria": "facturacion",
  "urgencia": "baja",
  "resumen": "El cliente consulta si la factura del mes actual ya se encuentra disponible.",
  "requiere_escalamiento": false
}

Objeto Python validado: categoria='facturacion' urgencia='baja' resumen='El cliente consulta si la factura del mes actual ya se encuentra disponible.' requiere_escalamiento=False


Notá que igual pasamos el JSON por `ConsultaSoporte.model_validate_json` aunque Bedrock ya garantiza el schema — no es redundante: valida también las reglas que un JSON Schema básico no expresa bien (en este caso ninguna, pero en general sí: por ejemplo relaciones entre campos, o validadores custom de Pydantic).

**🏠 Practicá**: cambiá `texto_libre` por una consulta de facturación de baja urgencia y confirmá que `categoria` y `urgencia` salen correctas sin que se lo digas explícitamente — el modelo infiere la clasificación, no la copia del texto.

## 3. Manejo de respuestas inválidas

No todos los proveedores tienen un mecanismo nativo de structured outputs (Anthropic directo, Google y Ollama no lo tienen igual de simple). Para esos casos, o como red de seguridad general, conviene un patrón universal: **pedir JSON en el prompt, parsear, validar con Pydantic, y si falla, reintentar mostrándole al modelo el error exacto** para que se corrija.

In [35]:
def generar_estructurado_con_reintentos(llamar_fn, prompt, schema_model, system=None, max_reintentos=3):
    """Patrón universal: sirve con cualquier función llamar_* (Bedrock, OpenAI, Anthropic, Google, Ollama)."""
    instruccion_base = (
        f"{prompt}\n\nRespondé ÚNICAMENTE con un JSON válido que cumpla este schema"
        f"(sin texto adicional, sin markdown):\n{schema_model.model_json_schema()}"
    )
    ultimo_error = None

    for intento in range(max_reintentos):
        instruccion = instruccion_base
        if ultimo_error:
            instruccion += f"\n\nTu respuesta anterior no cumplió el schema. Error: {ultimo_error}\nCorregí y devolvé SOLO el JSON."

        texto = llamar_fn(instruccion, system=system, temperature=0.0)  # temperature baja: queremos formato consistente, no creatividad
        try:
            return schema_model.model_validate_json(texto)
        except (ValidationError, json.JSONDecodeError) as e:
            ultimo_error = str(e)
            print(f"Intento {intento + 1} inválido, reintentando...")

    raise ValueError(f"No se obtuvo una respuesta válida tras {max_reintentos} intentos. Último error: {ultimo_error}")

# Prueba con Google, sin usar el mecanismo nativo (simula un proveedor sin structured outputs)
resultado = generar_estructurado_con_reintentos(
    llamar_google,
    f"Clasificá esta consulta de soporte: '{texto_libre}'",
    ConsultaSoporte,
)

print(resultado)

categoria='facturacion' urgencia='baja' resumen='El cliente consulta si la factura del mes actual ya se encuentra disponible.' requiere_escalamiento=False


In [8]:
def llamar_tramposo(prompt, system=None, temperature=0.7, max_tokens=500):
    return "¡Hola! Esto no es un JSON"

resultado = generar_estructurado_con_reintentos(
    llamar_tramposo,
    "Clasificá esta consulta: 'no me llegó la factura'",
    ConsultaSoporte,
    max_reintentos=2,
)
print(resultado)

Intento 1 inválido, reintentando...
Intento 2 inválido, reintentando...


ValueError: No se obtuvo una respuesta válida tras 2 intentos. Último error: 1 validation error for ConsultaSoporte
  Invalid JSON: expected value at line 1 column 1 [type=json_invalid, input_value='¡Hola! Esto no es un JSON', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/json_invalid

**🏠 Practicá**: bajá `max_reintentos` a 1 y modificá el prompt para pedir un campo que no existe en el schema (ej. "incluí también un campo 'telefono'") — el modelo probablemente va a devolver un campo extra o mal formado. Observá el error de Pydantic y confirmá que la función lo reporta con `raise ValueError`, no con un traceback críptico.

## 4. Tool calling

Tool calling es un caso particular de salida estructurada: en vez de pedirle al modelo un objeto de negocio, le das una lista de **herramientas disponibles** (funciones con su schema de argumentos) y el modelo decide *si* necesita alguna y con qué argumentos — vos ejecutás la función real y le devolvés el resultado.

El modelo **nunca ejecuta código** — solo pide "llamá a esta función con estos argumentos". La ejecución es responsabilidad de tu aplicación (esto es clave para seguridad, lo retomamos en la Clase 9 y la Clase 11).

In [9]:
# La "herramienta" real: una función Python común y corriente
def consultar_estado_ticket(ticket_id: str) -> dict:
    tickets_mock = {"T-001": "En progreso", "T-002": "Resuelto", "T-003": "Esperando al cliente"}
    return {"ticket_id": ticket_id, "estado": tickets_mock.get(ticket_id, "No encontrado")}


# La descripción de esa función que le mandamos al modelo (JSON Schema de sus argumentos)
tool_config = {
    "tools": [{
        "toolSpec": {
            "name": "consultar_estado_ticket",
            "description": "Consulta el estado actual de un ticket de soporte por su ID.",
            "inputSchema": {"json": {
                "type": "object",
                "properties": {
                    "ticket_id": {"type": "string", "description": "ID del ticket, ej. T-001"}
                },
                "required": ["ticket_id"],
            }},
        }
    }]
}

In [10]:
print(consultar_estado_ticket("T-002"))
print(consultar_estado_ticket("T-999"))  # ticket inexistente

{'ticket_id': 'T-002', 'estado': 'Resuelto'}
{'ticket_id': 'T-999', 'estado': 'No encontrado'}


In [11]:
herramienta_google = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="consultar_estado_ticket",
            description="Consulta el estado actual de un ticket de soporte por su ID.",
            parameters={
                "type": "object",
                "properties": {
                    "ticket_id": {"type": "string", "description": "ID del ticket, ej. T-001"}
                },
                "required": ["ticket_id"],
            },
        )
    ]
)

print(herramienta_google)

retrieval=None computer_use=None file_search=None google_search=None google_maps=None code_execution=None enterprise_web_search=None function_declarations=[FunctionDeclaration(
  description='Consulta el estado actual de un ticket de soporte por su ID.',
  name='consultar_estado_ticket',
  parameters=Schema(
    properties={
      'ticket_id': Schema(
        description='ID del ticket, ej. T-001',
        type=<Type.STRING: 'STRING'>
      )
    },
    required=[
      'ticket_id',
    ],
    type=<Type.OBJECT: 'OBJECT'>
  )
)] google_search_retrieval=None parallel_ai_search=None url_context=None mcp_servers=None exa_ai_search=None


In [12]:
pregunta = "¿Cuál es el estado del ticket T-999?"

config = types.GenerateContentConfig(tools=[herramienta_google])

respuesta = cliente.models.generate_content(
    model=model_id,
    contents=pregunta,
    config=config,
)

print(respuesta.function_calls)

[FunctionCall(
  args={
    'ticket_id': 'T-999'
  },
  id='call_3025562840242306783',
  name='consultar_estado_ticket'
)]


In [13]:
pedido = respuesta.function_calls[0]

print(pedido.name) 
print(pedido.args)

consultar_estado_ticket
{'ticket_id': 'T-999'}


In [14]:
resultado = consultar_estado_ticket(**pedido.args)
print(resultado)

{'ticket_id': 'T-999', 'estado': 'No encontrado'}


In [15]:
expediente = [
    types.Content(role="user", parts=[types.Part.from_text(text=pregunta)]),
    respuesta.candidates[0].content,
    types.Content(role="user", parts=[types.Part.from_function_response(name=pedido.name, response=resultado)]),
]

respuesta_final = cliente.models.generate_content(
    model=model_id,
    contents=expediente,
    config=config,
)

print(respuesta_final.text)

El ticket con ID **T-999** no fue encontrado en el sistema. Por favor, verifique que el número sea correcto.


In [ ]:
# BEDROCK — solo referencia, NO CORRER (versión Google más abajo)

messages = [{"role": "user", "content": [{"text": "¿Cuál es el estado del ticket T-002?"}]}]

response = bedrock.converse(modelId=model_id, messages=messages, toolConfig=tool_config)
messages.append(response["output"]["message"])

print("Stop reason:", response["stopReason"])

if response["stopReason"] == "tool_use":
    for block in response["output"]["message"]["content"]:
        if "toolUse" in block:
            tool = block["toolUse"]
            print(f"El modelo pide llamar a: {tool['name']}({tool['input']})")

            if tool["name"] == "consultar_estado_ticket":
                resultado = consultar_estado_ticket(**tool["input"])  # acá ejecutamos NOSOTROS la función real
                messages.append({
                    "role": "user",
                    "content": [{"toolResult": {"toolUseId": tool["toolUseId"], "content": [{"json": resultado}]}}],
                })

    # Segunda llamada: le devolvemos el resultado de la herramienta para que arme la respuesta final
    response_final = bedrock.converse(modelId=model_id, messages=messages, toolConfig=tool_config)
    print("\nRespuesta final:", response_final["output"]["message"]["content"][0]["text"])
else:
    print(response["output"]["message"]["content"][0]["text"])

Stop reason: tool_use
El modelo pide llamar a: consultar_estado_ticket({'ticket_id': 'T-002'})

Respuesta final: El estado actual del ticket **T-002** es: ✅ **Resuelto**.

Esto significa que el problema o solicitud asociada a este ticket ha sido atendida y cerrada satisfactoriamente. Si tienes alguna otra consulta o necesitas más detalles, ¡con gusto te ayudo!


In [ ]:
# BEDROCK — solo referencia, NO CORRER (versión Google más abajo)

messages

[{'role': 'user',
  'content': [{'text': '¿Cuál es el estado del ticket T-002?'}]},
 {'role': 'assistant',
  'content': [{'text': 'Voy a consultar el estado del ticket T-002 de inmediato.'},
   {'toolUse': {'toolUseId': 'tooluse_B5Xrjcv6c0nRxhRlWnZTSM',
     'name': 'consultar_estado_ticket',
     'input': {'ticket_id': 'T-002'},
     'type': 'tool_use'}}]},
 {'role': 'user',
  'content': [{'toolResult': {'toolUseId': 'tooluse_B5Xrjcv6c0nRxhRlWnZTSM',
     'content': [{'json': {'ticket_id': 'T-002', 'estado': 'Resuelto'}}]}}]}]

**🏠 Practicá**: preguntá por un ticket que no existe (ej. "T-999") y confirmá que la respuesta final del modelo refleja el "No encontrado" que devuelve la función — no inventa un estado. Después agregá una segunda herramienta mock (ej. `escalar_ticket(ticket_id, motivo)`) y preguntá algo que dispare esa herramienta en vez de la primera.

## 5. Timeouts, reintentos y backoff

En la Clase 2 armamos `generar_con_reintentos` con backoff exponencial fijo (1s, 2s, 4s...). Dos mejoras para producción:

- **Timeout explícito**: sin él, una llamada colgada puede bloquear tu aplicación indefinidamente.
- **Jitter**: sumar un poco de aleatoriedad al backoff evita que muchos clientes reintenten exactamente al mismo tiempo tras un error masivo (efecto "estampida").

In [ ]:
# BEDROCK — solo referencia, NO CORRER (versión Google más abajo)


from botocore.config import Config
from botocore.exceptions import ClientError

# Cliente con timeouts explícitos y sin el retry automático de boto3 (lo manejamos nosotros)
bedrock_prod = boto3.client(
    "bedrock-runtime",
    region_name=os.environ.get("AWS_REGION", "us-east-1"),
    config=Config(connect_timeout=5, read_timeout=15, retries={"max_attempts": 0}),
)


def generar_con_reintentos(bedrock_client, model_id, messages, system=None, inference_config=None, max_reintentos=3):
    kwargs = {"modelId": model_id, "messages": messages}
    if system:
        kwargs["system"] = system
    if inference_config:
        kwargs["inferenceConfig"] = inference_config

    for intento in range(max_reintentos):
        try:
            return bedrock_client.converse(**kwargs)
        except ClientError as err:
            codigo = err.response["Error"]["Code"]
            if codigo == "ThrottlingException" and intento < max_reintentos - 1:
                espera = (2 ** intento) + random.uniform(0, 1)  # backoff exponencial + jitter
                print(f"Rate limit, reintentando en {espera:.1f}s...")
                time.sleep(espera)
                continue
            if codigo == "ValidationException":
                raise ValueError(f"Request inválido: {err.response['Error']['Message']}") from err
            raise
    raise RuntimeError("Se agotaron los reintentos por throttling.")


resp = generar_con_reintentos(bedrock_prod, model_id, messages=[{"role": "user", "content": [{"text": "Decí 'ok'."}]}])
print(resp["output"]["message"]["content"][0]["text"])

ok


In [16]:
for intento in range(4):
    espera = (2 ** intento) + random.uniform(0, 1)
    print(f"Intento {intento}: esperaría {espera:.1f} segundos")

Intento 0: esperaría 1.1 segundos
Intento 1: esperaría 2.4 segundos
Intento 2: esperaría 4.8 segundos
Intento 3: esperaría 8.1 segundos


In [17]:
from google.genai import errors

CODIGOS_REINTENTABLES = [429, 500, 502, 503, 504]

def llamar_google_con_reintentos(prompt, max_reintentos=3):
    for intento in range(max_reintentos):
        try:
            return llamar_google(prompt)
        except errors.APIError as err:
            if err.code in CODIGOS_REINTENTABLES and intento < max_reintentos - 1:
                espera = (2 ** intento) + random.uniform(0, 1)
                print(f"Error {err.code}, reintentando en {espera:.1f} segundos...")
                time.sleep(espera)
            else:
                raise

In [18]:
print(llamar_google_con_reintentos("Decí 'ok'."))

Ok


In [19]:
# ERROR A PROPÓSITO: timeout de 10 ms para ver qué pasa cuando se corta


cliente_apurado = genai.Client(
    api_key=os.environ["GOOGLE_API_KEY"],
    http_options=types.HttpOptions(timeout=10),
)

respuesta = cliente_apurado.models.generate_content(
    model=model_id,
    contents="Decí 'ok'.",
)
print(respuesta.text)

ConnectTimeout: _ssl.c:993: The handshake operation timed out

**🏠 Practicá**: bajá `read_timeout` a 0.01 segundos y volvé a correr la celda anterior — debería fallar con un timeout de lectura. Envolvé la llamada en un `try/except` que capture ese error específico (`botocore.exceptions.ReadTimeoutError`) y decidí si tiene sentido reintentar (sí: un timeout es, en general, reintentable).

## 6. Observabilidad básica: tokens, costo y latencia

Sin esto, no hay forma de responder "¿cuánto nos está costando esto?" ni de detectar una llamada anormalmente lenta. Registrarlo desde el día 1 es mucho más barato que agregarlo después.

In [ ]:
# BEDROCK — solo referencia, NO CORRER

# Valores de ejemplo — el precio real cambia con el tiempo, verificar en la consola de Bedrock antes de usar en producción
PRECIO_POR_MILLON_USD = {"input": 3.0, "output": 15.0}

def estimar_costo_usd(usage: dict) -> float:
    costo_input = usage["inputTokens"] / 1_000_000 * PRECIO_POR_MILLON_USD["input"]
    costo_output = usage["outputTokens"] / 1_000_000 * PRECIO_POR_MILLON_USD["output"]
    return costo_input + costo_output


registro_llamadas = []  # en una app real, esto va a un log estructurado o una base de datos (Clase 11)

def llamar_con_registro(bedrock_client, model_id, messages, **kwargs):
    resp = generar_con_reintentos(bedrock_client, model_id, messages, **kwargs)
    registro_llamadas.append({
        "input_tokens": resp["usage"]["inputTokens"],
        "output_tokens": resp["usage"]["outputTokens"],
        "costo_usd": estimar_costo_usd(resp["usage"]),
        "latencia_ms": resp["metrics"]["latencyMs"],
    })
    return resp

resp = llamar_con_registro(bedrock_prod, model_id, messages=[{"role": "user", "content": [{"text": "Explicá qué es un embedding en 1 oración."}]}])
print(resp["output"]["message"]["content"][0]["text"])
print("\nRegistro:", registro_llamadas[-1])

In [20]:
respuesta = cliente.models.generate_content(
    model=model_id,
    contents="Explicá qué es un embedding en 1 oración.",
)

print(respuesta.text)
print()
print("Tokens de entrada:", respuesta.usage_metadata.prompt_token_count)
print("Tokens de salida:", respuesta.usage_metadata.candidates_token_count)
print("Tokens en total:", respuesta.usage_metadata.total_token_count)


Un **embedding** es una representación matemática en forma de números (vectores) de conceptos como palabras, imágenes o textos, que permite a las computadoras entender sus significados y relaciones según qué tan cerca están unos de otros en el espacio digital.

Tokens de entrada: 13
Tokens de salida: 50
Tokens en total: 63


In [21]:
# Precios DE EJEMPLO (los mismos del notebook). Los reales de Gemini están en la página de precios de Google.
PRECIO_POR_MILLON_USD = {"input": 3.0, "output": 15.0}

def estimar_costo_usd(uso):
    costo_entrada = uso.prompt_token_count / 1_000_000 * PRECIO_POR_MILLON_USD["input"]
    costo_salida = uso.candidates_token_count / 1_000_000 * PRECIO_POR_MILLON_USD["output"]
    return costo_entrada + costo_salida

print(estimar_costo_usd(respuesta.usage_metadata))

0.000789


In [22]:
inicio = time.time()

respuesta = cliente.models.generate_content(
    model=model_id,
    contents="Explicá qué es un embedding en 1 oración.",
)

fin = time.time()

latencia_ms = (fin - inicio) * 1000
print(f"Tardó {latencia_ms:.0f} milisegundos")

Tardó 1014 milisegundos


In [23]:
registro_llamadas = []

def llamar_con_registro(prompt):
    inicio = time.time()
    respuesta = cliente.models.generate_content(model=model_id, contents=prompt)
    fin = time.time()

    registro_llamadas.append({
        "tokens_entrada": respuesta.usage_metadata.prompt_token_count,
        "tokens_salida": respuesta.usage_metadata.candidates_token_count,
        "costo_usd": estimar_costo_usd(respuesta.usage_metadata),
        "latencia_ms": (fin - inicio) * 1000,
    })

    return respuesta.text

In [24]:
print(llamar_con_registro("Explicá qué es un embedding en 1 oración."))
print(llamar_con_registro("¿Cuánto es 2 + 2? Respondé solo el número."))
print()
print(registro_llamadas)

Un **embedding** es la representación matemática de un objeto (como una palabra o una imagen) en forma de números, diseñada para que las computadoras entiendan su significado y similitud con otros objetos según la cercanía entre esos números.
4

[{'tokens_entrada': 13, 'tokens_salida': 49, 'costo_usd': 0.000774, 'latencia_ms': 876.4750957489014}, {'tokens_entrada': 16, 'tokens_salida': 1, 'costo_usd': 6.3e-05, 'latencia_ms': 646.0878849029541}]


In [25]:
print(estimar_costo_usd(respuesta.usage_metadata) * 10000)

7.59


**🏠 Practicá**: hacé 5 llamadas distintas con `llamar_con_registro` y al final calculá el costo total y la latencia promedio a partir de `registro_llamadas` (con una simple suma/promedio sobre la lista de dicts). Esto es literalmente lo que vas a necesitar para el informe de costos del TP Final (Clase 12).

## 7. Streaming

Para respuestas largas, esperar a que termine toda la generación antes de mostrar nada da una sensación de latencia alta. El streaming muestra el texto a medida que se genera, token a token.

In [ ]:
# BEDROCK — solo referencia, NO CORRER

start=time.time()
stream_response = bedrock.converse_stream(
    modelId=model_id,
    messages=[{"role": "user", "content": [{"text": "Listá 5 buenas prácticas de seguridad para agentes de IA, con una oración de explicación cada una."}]}],
)
print(stream_response)
for event in stream_response["stream"]:
    if "contentBlockDelta" in event:
        print(time.time()-start)
        print(event["contentBlockDelta"]["delta"]["text"], end="", flush=True)
    if "metadata" in event and "usage" in event["metadata"]:
        print(f"\n\n[tokens: {event['metadata']['usage']}]")
final = time.time()-start
print(final)

In [26]:
stream = cliente.models.generate_content_stream(
    model=model_id,
    contents="Listá 5 buenas prácticas de seguridad para agentes de IA, con una oración de explicación cada una.",
)

for pedazo in stream:
    print(pedazo.text, end="", flush=True)

Aquí tienes 5 buenas prácticas de seguridad para agentes de IA:

1. **Principio de mínimo privilegio:** Limita los permisos y accesos del agente solo a las herramientas, bases de datos y APIs estrictamente necesarias para cumplir su función específica.
2. **Validación y sanitización de entradas:** Implementa filtros robustos para neutralizar intentos de inyección deprompts (*prompt injection*) y asegurar que los datos ingresados por el usuario sean seguros.
3. **Monitoreo y registro de actividades (*Logging*):** Registra y audita todas las interacciones, decisiones y llamadas a herramientas realizadas por el agente para detectar comportamientos anómalos a tiempo.
4. **Human-in-the-loop (Humano en el circuito):** Exige la aprobación humana obligatoria antes de que el agente ejecute acciones críticas, como realizar transacciones financieras, borrar datos o enviar comunicaciones externas.
5. **Alineación y barandillas de seguridad (*Guardrails*):** Define límites estrictos mediante restri

In [27]:
inicio = time.time()
primer_pedazo = None

stream = cliente.models.generate_content_stream(
    model=model_id,
    contents="Listá 5 buenas prácticas de seguridad para agentes de IA, con una oración de explicación cada una.",
)

for pedazo in stream:
    if primer_pedazo is None:
        primer_pedazo = time.time() - inicio
    print(pedazo.text, end="", flush=True)

total = time.time() - inicio

print()
print(f"Primer pedazo: {primer_pedazo:.2f} segundos")
print(f"Respuesta completa: {total:.2f} segundos")

Aquí tienes 5 buenas prácticas de seguridad para agentes de IA:

1. **Principio de mínimo privilegio:** Restringe los permisos de acceso del agente a bases de datos y herramientas externas, concediéndole únicamente los recursos estrictamente necesarios para cumplir su función.
2. **Validación y sanitización de entradas:** Implementa filtros rigurosos para desinfectar tanto el *prompt* del usuario como los datos recuperados de fuentes externas, previniendo así ataques de inyección de instrucciones (*prompt injection*).
3. **Control humano en el bucle (*Human-in-the-loop*):** Exige la supervisión y aprobación humana antes de que el agente ejecute acciones críticas o irreversibles, como realizar transacciones financieras o modificar datos sensibles.
4. **Enmascaramiento de datos sensibles:** Utiliza técnicas para detectar y anonimizar información personal o confidencial antes de enviarla a los modelos de lenguaje, protegiendo así la privacidad.
5. **Monitoreo y registro de auditoría (*Log

In [28]:
inicio = time.time()

respuesta = cliente.models.generate_content(
    model=model_id,
    contents="Listá 5 buenas prácticas de seguridad para agentes de IA, con una oración de explicación cada una.",
)

total_sin_stream = time.time() - inicio

print(respuesta.text)
print(f"Sin streaming, ves la primera palabra a los: {total_sin_stream:.2f} segundos")

Aquí tienes 5 buenas prácticas de seguridad para agentes de IA:

1. **Principio de mínimo privilegio:** Restringe los permisos de acceso del agente a bases de datos, APIs y herramientas solo a lo estrictamente necesario para cumplir su función específica.
2. **Validación y sanitización de entradas:** Implementa filtros rigurosos en las entradas de los usuarios para prevenir ataques de inyección de prompts (*prompt injection*) y la ejecución maliciosa de código.
3. **Monitoreo y registro de auditoría (*Logging*):** Registra todas las interacciones, decisiones y llamadas a herramientas del agente para facilitar la detección de anomalías y la revisión forense ante incidentes.
4. **Human-in-the-loop (Humano en el circuito):** Exige la aprobación humana obligatoria antes de que el agente ejecute acciones críticas, como realizar transacciones financieras, borrar datos o modificar configuraciones del sistema.
5. **Aislamiento del entorno (*Sandboxing*):** Ejecuta el código generado por el age

In [ ]:
# BEDROCK — solo referencia, NO CORRER

stream_response

{'ResponseMetadata': {'RequestId': '23a0002d-ca60-4a74-ab10-079f0ede3288',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Wed, 07 Oct 2026 00:40:10 GMT',
   'content-type': 'application/vnd.amazon.eventstream',
   'transfer-encoding': 'chunked',
   'connection': 'keep-alive',
   'x-amzn-requestid': '23a0002d-ca60-4a74-ab10-079f0ede3288'},
  'RetryAttempts': 0},
 'stream': <botocore.eventstream.EventStream at 0x7182fe00d270>}

In [ ]:
# BEDROCK — solo referencia, NO CORRER

stream_response

{'ResponseMetadata': {'RequestId': '38b271b1-05c9-4e63-8c63-5d67101fe456',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Wed, 07 Oct 2026 00:37:41 GMT',
   'content-type': 'application/vnd.amazon.eventstream',
   'transfer-encoding': 'chunked',
   'connection': 'keep-alive',
   'x-amzn-requestid': '38b271b1-05c9-4e63-8c63-5d67101fe456'},
  'RetryAttempts': 0},
 'stream': <botocore.eventstream.EventStream at 0x757b55ae04c0>}

**🏠 Practicá**: medí con `time.time()` cuánto tarda en llegar el **primer** chunk de texto (time to first token) vs. cuánto tarda la respuesta completa con `converse` normal (sin streaming) para el mismo prompt. Esa diferencia es la que justifica usar streaming en una UI interactiva.

## 8. Abstracción de proveedores

Hasta acá todo el notebook (y el `LLMClient` de la Clase 2) llama directo a Bedrock, con un `if/elif` por proveedor en el mejor de los casos. Eso no escala: cada vez que agregás lógica nueva (reintentos, structured outputs, registro de costos) hay que replicarla en cada rama.

La solución: una **interfaz común** (clase abstracta) que cada proveedor implementa. El resto de la aplicación programa contra la interfaz, no contra un SDK específico.

In [29]:
from abc import ABC, abstractmethod


class LLMProvider(ABC):
    """Interfaz común para cualquier proveedor de modelos de lenguaje."""

    @abstractmethod
    def generate(self, prompt: str, system: str = None, temperature: float = 0.7, max_tokens: int = 500) -> str:
        """Devuelve el texto generado por el modelo."""
        ...

    def generate_structured(self, prompt: str, schema_model, system: str = None):
        """Implementación por defecto: sirve para CUALQUIER proveedor, aunque no tenga
        soporte nativo de structured outputs. Un proveedor que sí lo tenga (como Bedrock)
        puede sobreescribir este método con una versión más eficiente (ver BedrockProvider)."""
        return generar_estructurado_con_reintentos(self.generate, prompt, schema_model, system=system)

In [30]:
proveedor = LLMProvider()

TypeError: Can't instantiate abstract class LLMProvider without an implementation for abstract method 'generate'

In [31]:
class BedrockProvider(LLMProvider):
    def __init__(self):
        self.client = boto3.client(
            "bedrock-runtime",
            region_name=os.environ.get("AWS_REGION", "us-east-1"),
            config=Config(connect_timeout=5, read_timeout=15, retries={"max_attempts": 0}),
        )
        self.model_id = os.environ.get("BEDROCK_MODEL_ID_CLAUDE", "global.anthropic.claude-sonnet-4-6")

    def generate(self, prompt, system=None, temperature=0.7, max_tokens=500):
        messages = [{"role": "user", "content": [{"text": prompt}]}]
        system_param = [{"text": system}] if system else None
        inference_config = {"temperature": temperature, "maxTokens": max_tokens}
        resp = generar_con_reintentos(self.client, self.model_id, messages, system=system_param, inference_config=inference_config)
        return resp["output"]["message"]["content"][0]["text"]

    def generate_structured(self, prompt, schema_model, system=None):
        # Override: usa el mecanismo nativo de Bedrock (sección 2) en vez del reintento genérico
        schema = schema_model.model_json_schema()
        schema["additionalProperties"] = False
        kwargs = {
            "modelId": self.model_id,
            "messages": [{"role": "user", "content": [{"text": prompt}]}],
            "outputConfig": {"textFormat": {"type": "json_schema", "structure": {"jsonSchema": {
                "name": schema_model.__name__, "description": schema_model.__doc__ or "", "schema": json.dumps(schema),
            }}}},
        }
        if system:
            kwargs["system"] = [{"text": system}]
        resp = self.client.converse(**kwargs)
        raw = resp["output"]["message"]["content"][0]["text"]
        return schema_model.model_validate_json(raw)

Los demás proveedores quedan como ejercicio — solo necesitan implementar `generate()` (ya lo tenés resuelto de la Clase 2, sección "Tu proveedor"); `generate_structured()` viene gratis de la clase base.

In [32]:
class AnthropicProvider(LLMProvider):
    def __init__(self):
        import anthropic
        self.client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
        self.model_id = os.environ.get("ANTHROPIC_MODEL_ID", "claude-sonnet-4-6")

    def generate(self, prompt, system=None, temperature=0.7, max_tokens=500):
        # TODO: reutilizar la lógica de llamar_anthropic (Clase 2, sección 2)
        raise NotImplementedError


class OpenAIProvider(LLMProvider):
    def __init__(self):
        from openai import OpenAI
        self.client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
        self.model_id = os.environ.get("OPENAI_MODEL_ID", "gpt-4o")

    def generate(self, prompt, system=None, temperature=0.7, max_tokens=500):
        # TODO: reutilizar la lógica de llamar_openai (Clase 2, sección 2)
        raise NotImplementedError


class GoogleProvider(LLMProvider):
    def __init__(self):
        from google import genai
        self.client = genai.Client(
            api_key=os.environ["GOOGLE_API_KEY"],
            http_options=types.HttpOptions(timeout=30000),
        )
        self.model_id = os.environ.get("GOOGLE_MODEL_ID", "gemini-3.5-flash-lite")

    def generate(self, prompt, system=None, temperature=0.7, max_tokens=500):
        config = types.GenerateContentConfig(
            system_instruction=system,
            temperature=temperature,
            max_output_tokens=max_tokens,
        )
        respuesta = self.client.models.generate_content(
            model=self.model_id,
            contents=prompt,
            config=config,
        )
        return respuesta.text


class OllamaProvider(LLMProvider):
    def __init__(self):
        self.model_id = os.environ.get("OLLAMA_MODEL_ID", "llama3.1")

    def generate(self, prompt, system=None, temperature=0.7, max_tokens=500):
        # TODO: reutilizar la lógica de llamar_ollama (Clase 2, sección 2)
        raise NotImplementedError


def get_provider() -> LLMProvider:
    """Factory: devuelve la instancia del proveedor configurado en LLM_PROVIDER."""
    proveedores = {
        "bedrock": BedrockProvider,
        "anthropic": AnthropicProvider,
        "openai": OpenAIProvider,
        "google": GoogleProvider,
        "ollama": OllamaProvider,
    }
    nombre = os.environ.get("LLM_PROVIDER", "bedrock")
    if nombre not in proveedores:
        raise ValueError(f"Proveedor desconocido: {nombre}")
    return proveedores[nombre]()


# Prueba: esto funciona hoy con Bedrock. Cuando completes los TODO de arriba, va a funcionar
# igual (sin cambiar esta celda) con cualquier otro proveedor que configures en LLM_PROVIDER.
llm = get_provider()
print(llm.generate("Decí 'funciona' y nada más."))

funciona


In [33]:
resultado = llm.generate_structured(
    "Clasificá esta consulta: 'no me llegó la factura'",
    ConsultaSoporte,
)
print(resultado)

categoria='facturacion' urgencia='media' resumen='El cliente reporta que no ha recibido su factura.' requiere_escalamiento=False


**🏠 Practicá**: completá al menos un `TODO` (el de tu proveedor real, si no es Bedrock) y confirmá que `get_provider().generate(...)` funciona sin tocar ninguna otra celda del notebook. Después probá `get_provider().generate_structured("Clasificá esta consulta: 'no me llegó la factura'", ConsultaSoporte)` — con Bedrock usa el mecanismo nativo (sección 2); con cualquier otro proveedor, usa el reintento genérico (sección 3) automáticamente, sin que hayas escrito nada extra para eso.

## 9. Laboratorio: servicio de extracción estructurada

**🏠 Tarea de la semana — entregar/completar antes de la Clase 4.** El docente presenta la consigna y muestra el patrón en vivo, pero la resolución es trabajo individual asincrónico.

**Checkpoint del proyecto integrador (Clase 3)**: el `LLMClient` de la Clase 2, ahora detrás de la interfaz `LLMProvider`, más un servicio que transforma texto libre en objetos validados.

### Consigna

1. Terminá de completar los `TODO` de `AnthropicProvider` / `OpenAIProvider` / `GoogleProvider` / `OllamaProvider` — al menos el de **tu** proveedor.
2. Definí un modelo Pydantic propio (no `ConsultaSoporte`) relevante para el dominio del TP integrador — por ejemplo, una clasificación de la intención de una pregunta sobre la documentación técnica del corpus (`tipo_pregunta`, `requiere_busqueda_en_documentacion: bool`, `resumen`).
3. Escribí una función `clasificar(texto: str) -> TuModelo` que use `get_provider().generate_structured(...)` para resolverlo.
4. Probala con al menos 3 textos de entrada distintos, incluyendo uno ambiguo o mal escrito, y confirmá que **siempre** devuelve un objeto válido o levanta una excepción clara (nunca un `dict` sin validar ni un texto crudo).

### Criterio de éxito

- El servicio nunca deja pasar un objeto que no cumple tu schema Pydantic — o devuelve el objeto validado, o falla explícitamente.
- Funciona con el proveedor configurado en tu `.env`, sin credenciales hardcodeadas.
- Podés explicar en una frase la diferencia entre usar `generate_structured` con Bedrock (nativo) y con cualquier otro proveedor (reintento + prompt) — sin que el código que llama a `clasificar()` necesite saberlo.

In [43]:
class IntencionConsulta(BaseModel):
    tipo_pregunta: Literal["concepto", "como_hacer", "error", "fuera_de_tema"] = Field(
        ...,
        description="concepto: qué es algo o cómo funciona. como_hacer: cómo lograr algo con código. error: un problema o resultado inesperado. fuera_de_tema: no tiene que ver con JavaScript.",
    )
    requiere_busqueda_en_documentacion: bool = Field(
        ...,
                description="True para toda pregunta sobre JavaScript, aunque sepas la respuesta. False solo si es fuera_de_tema.",

    )
    resumen: str = Field(..., max_length=200, description="La pregunta resumida en una oración.")

SYSTEM_CLASIFICADOR = (
    "Sos un clasificador de preguntas para un asistente que responde usando la documentación de JavaScript de MDN. "
    "Categorías: "
    "concepto = el usuario quiere entender qué es algo o cómo funciona. "
    "como_hacer = quiere lograr algo concreto con código. "
    "error = describe un problema, un error o un resultado inesperado. "
    "fuera_de_tema = la pregunta no tiene que ver con JavaScript. "
    "Regla: toda pregunta sobre JavaScript (concepto, como_hacer o error) requiere búsqueda en la documentación, "
    "aunque vos sepas la respuesta. Solo fuera_de_tema no requiere búsqueda."
)

def clasificar(texto: str) -> IntencionConsulta:
    if not texto or not texto.strip():
        raise ValueError("El texto a clasificar está vacío.")
    proveedor = get_provider()
    return proveedor.generate_structured(
        f"Clasificá esta pregunta de un usuario sobre JavaScript: '{texto}'",
        IntencionConsulta,
        system=SYSTEM_CLASIFICADOR,
    )

In [37]:
ejemplo = IntencionConsulta(
    tipo_pregunta="concepto",
    requiere_busqueda_en_documentacion=True,
    resumen="El usuario pregunta qué es un closure.",
)
print(ejemplo)

tipo_pregunta='concepto' requiere_busqueda_en_documentacion=True resumen='El usuario pregunta qué es un closure.'


In [39]:
resultado = clasificar("¿Qué es una promesa en JavaScript?")
print(resultado)

tipo_pregunta='concepto' requiere_busqueda_en_documentacion=True resumen='Consulta sobre la definición y funcionamiento de una promesa en JavaScript.'


In [42]:
pruebas = [
    "como ago para recorer un aray??",
    "me sale undefined cuando llamo a mi funcion y no entiendo xq",
    "¿Quién ganó el mundial 2022?",
]

for texto in pruebas:
    print("Pregunta:", texto)
    print(clasificar(texto))
    print()

Pregunta: como ago para recorer un aray??
tipo_pregunta='como_hacer' requiere_busqueda_en_documentacion=True resumen='El usuario quiere saber cómo recorrer un array en JavaScript.'

Pregunta: me sale undefined cuando llamo a mi funcion y no entiendo xq
tipo_pregunta='error' requiere_busqueda_en_documentacion=True resumen="El usuario recibe 'undefined' al llamar a una función y necesita entender por qué ocurre este resultado inesperado."

Pregunta: ¿Quién ganó el mundial 2022?
tipo_pregunta='fuera_de_tema' requiere_busqueda_en_documentacion=False resumen='El usuario pregunta quién ganó el mundial de fútbol 2022.'



In [44]:
clasificar("     ")

ValueError: El texto a clasificar está vacío.

## Cierre

- Hoy el `LLMClient` de la Clase 2 se convirtió en algo bastante más sólido: interfaz común (`LLMProvider`), salidas estructuradas y validadas, tool calling, timeouts, reintentos con jitter, registro de costos y streaming.
- Esto **es** el primer entregable serio del TP integrador — de acá en adelante (RAG, agentes, despliegue) todo se construye sobre esta base, no se vuelve a escribir.
- Antes de la próxima clase: dejar el laboratorio de la sección 9 funcionando con tu proveedor.
- Próxima clase: embeddings e ingesta documental — empezamos a preparar el corpus para la recuperación semántica.